# 로컬 YOLO11n 음식 위치 탐지 학습

이 노트북은 음식 이미지와 `metadata.csv`의 Bounding Box를 이용해 로컬 GPU에서 음식 위치 탐지 모델을 학습·평가합니다.

In [ ]:
from pathlib import Path
import subprocess, sys, torch
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
assert (PROJECT_ROOT / 'scripts/prepare_yolo_food_dataset.py').is_file(), PROJECT_ROOT
DATASET_SOURCE_ROOT = Path(r'C:\dev\final_1_team\data\processed\aihub_food_image_text\v2\food_description_data')
assert (DATASET_SOURCE_ROOT / 'metadata.csv').is_file(), DATASET_SOURCE_ROOT
print('Python:', sys.executable)
print('CUDA 사용 가능:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음')

In [ ]:
# 최초 1회만 설치합니다. 현재 노트북 커널과 같은 가상환경에서 실행하세요.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-local.txt'], check=True)
from ultralytics import YOLO
print('Ultralytics 확인 완료')

In [ ]:
OUTPUT_ROOT = PROJECT_ROOT / 'data/training/yolo_food_detection'
assert not OUTPUT_ROOT.exists(), f'기존 결과를 보존하기 위해 중단했습니다: {OUTPUT_ROOT}'
subprocess.run([sys.executable, '-m', 'scripts.prepare_yolo_food_dataset', '--source-root', str(DATASET_SOURCE_ROOT), '--output-root', str(OUTPUT_ROOT)], cwd=PROJECT_ROOT, check=True)
print((OUTPUT_ROOT / 'dataset_audit.txt').read_text(encoding='utf-8'))

In [ ]:
RUN_NAME = 'yolo11n_food_v1'
DEVICE = '0' if torch.cuda.is_available() else 'cpu'
subprocess.run([sys.executable, '-m', 'scripts.train_yolo11n_food_detector', '--data', str(OUTPUT_ROOT / 'dataset.yaml'), '--epochs', '100', '--imgsz', '960', '--device', DEVICE, '--project', 'runs/yolo_food_detector', '--name', RUN_NAME], cwd=PROJECT_ROOT, check=True)
BEST_WEIGHTS = PROJECT_ROOT / 'runs/yolo_food_detector' / RUN_NAME / 'weights/best.pt'
assert BEST_WEIGHTS.is_file(), BEST_WEIGHTS

In [ ]:
subprocess.run([sys.executable, '-m', 'scripts.evaluate_yolo11n_food_detector', '--weights', str(BEST_WEIGHTS), '--data', str(OUTPUT_ROOT / 'dataset.yaml'), '--imgsz', '960', '--device', DEVICE, '--project', 'runs/yolo_food_detector_evaluation', '--name', RUN_NAME], cwd=PROJECT_ROOT, check=True)
METRICS_PATH = PROJECT_ROOT / 'runs/yolo_food_detector_evaluation' / RUN_NAME / 'metrics.json'
print(METRICS_PATH.read_text(encoding='utf-8'))